In [ ]:
%pip -q install datasets

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset
from PIL import Image
import json

POTATO_REPO = "Project-AgML/potato_leaf_disease_classification"
POTATO_RAW_DIR = Path("/content/potato_raw")
COMPLETE_MARKER = POTATO_RAW_DIR / "_download_complete.json"

if COMPLETE_MARKER.exists():
    print("Potato images already downloaded:", POTATO_RAW_DIR)
    saved_counts = json.loads(COMPLETE_MARKER.read_text())
    print("Images by class:")
    for class_name, count in saved_counts["counts"].items():
        print(f"{class_name}: {count}")
    print("Total:", saved_counts["total"])
else:
    print("Downloading Potato dataset from Hugging Face...")
    potato_ds = load_dataset(POTATO_REPO, split="train")

    print("Dataset rows:", len(potato_ds))
    print("Dataset columns:", potato_ds.column_names)
    print("Dataset features:", potato_ds.features)

    if "image" not in potato_ds.column_names or "label" not in potato_ds.column_names:
        raise ValueError(
            f"Expected image and label columns; found: {potato_ds.column_names}"
        )

    label_feature = potato_ds.features["label"]
    if not hasattr(label_feature, "names") or not label_feature.names:
        raise ValueError("Could not read the dataset's class names from its label feature.")

    class_names = label_feature.names
    counts = Counter()

    for index, row in enumerate(tqdm(potato_ds, desc="Saving Potato images")):
        class_name = class_names[int(row["label"])]
        image = row["image"]

        if not isinstance(image, Image.Image):
            raise TypeError(
                f"Expected a decoded PIL image, got {type(image)} at row {index}"
            )

        class_dir = POTATO_RAW_DIR / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        image_path = class_dir / f"potato_{index:05d}.png"
        image.convert("RGB").save(image_path, format="PNG")
        counts[class_name] += 1

    download_info = {
        "source": POTATO_REPO,
        "split": "train",
        "license": "CC-BY-4.0",
        "total": sum(counts.values()),
        "counts": dict(counts),
    }
    COMPLETE_MARKER.write_text(json.dumps(download_info, indent=2))

    print("\nSaved images to:", POTATO_RAW_DIR)
    print("Images by class:")
    for class_name in class_names:
        print(f"{class_name}: {counts[class_name]}")
    print("Total images:", sum(counts.values()))

README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  381MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  373MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3076 [00:00<?, ? examples/s]

Dataset rows: 3076
Dataset columns: ['image', 'label']
Dataset features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['Bacteria', 'Fungi', 'Healthy', 'Nematode', 'Pest', 'Phytopthora', 'Virus'])}


Saving Potato images:   0%|          | 0/3076 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset, Image as HFImage
import json

POTATO_REPO = "Project-AgML/potato_leaf_disease_classification"
POTATO_RAW_DIR = Path("/content/potato_raw_fast")
COMPLETE_MARKER = POTATO_RAW_DIR / "_download_complete.json"

if COMPLETE_MARKER.exists():
    saved_counts = json.loads(COMPLETE_MARKER.read_text())
    print("Fast image copy already exists:", POTATO_RAW_DIR)
    print("Counts:", saved_counts["counts"])
    print("Total:", saved_counts["total"])
else:
    # Reuse potato_ds already loaded in the interrupted cell.
    if "potato_ds" not in globals():
        print("Loading from the Hugging Face cache...")
        potato_ds = load_dataset(POTATO_REPO, split="train")

    class_names = potato_ds.features["label"].names

    # Ask Datasets for stored bytes instead of decoding images to PIL.
    raw_ds = potato_ds.cast_column("image", HFImage(decode=False))
    counts = Counter()

    for index, row in enumerate(tqdm(raw_ds, desc="Copying original image bytes")):
        class_name = class_names[int(row["label"])]
        image_data = row["image"]

        if not isinstance(image_data, dict):
            raise TypeError(
                f"Expected image bytes/path at row {index}; got {type(image_data)}"
            )

        image_bytes = image_data.get("bytes")
        source_path = image_data.get("path")

        if image_bytes is None and source_path and Path(source_path).is_file():
            image_bytes = Path(source_path).read_bytes()

        if image_bytes is None:
            raise RuntimeError(f"No image bytes available for dataset row {index}")

        extension = Path(source_path or "").suffix.lower()
        if extension not in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            if image_bytes.startswith(b"\xff\xd8\xff"):
                extension = ".jpg"
            elif image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
                extension = ".png"
            elif image_bytes[:4] == b"RIFF" and image_bytes[8:12] == b"WEBP":
                extension = ".webp"
            else:
                extension = ".jpg"

        class_dir = POTATO_RAW_DIR / class_name
        class_dir.mkdir(parents=True, exist_ok=True)

        image_path = class_dir / f"potato_{index:05d}{extension}"
        image_path.write_bytes(image_bytes)
        counts[class_name] += 1

    download_info = {
        "source": POTATO_REPO,
        "split": "train",
        "license": "CC-BY-4.0",
        "total": sum(counts.values()),
        "counts": dict(counts),
    }
    POTATO_RAW_DIR.mkdir(parents=True, exist_ok=True)
    COMPLETE_MARKER.write_text(json.dumps(download_info, indent=2))

    print("\nSaved original image files to:", POTATO_RAW_DIR)
    for class_name in class_names:
        print(f"{class_name}: {counts[class_name]}")
    print("Total images:", sum(counts.values()))

Copying original image bytes:   0%|          | 0/3076 [00:00<?, ?it/s]


Saved original image files to: /content/potato_raw_fast
Bacteria: 569
Fungi: 748
Healthy: 201
Nematode: 68
Pest: 611
Phytopthora: 347
Virus: 532
Total images: 3076


In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd

POTATO_RAW_DIR = Path("/content/potato_raw_fast")
POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
POTATO_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

records = []
unreadable = []

image_paths = [
    path for path in POTATO_RAW_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
]

for path in tqdm(image_paths, desc="Checking Potato images"):
    label = path.parent.name

    try:
        with Image.open(path) as image:
            image.verify()

        with Image.open(path) as image:
            width, height = image.size

        records.append({
            "image_path": str(path),
            "label": label,
            "width": width,
            "height": height,
        })
    except Exception as error:
        unreadable.append({
            "image_path": str(path),
            "error": str(error),
        })

potato_manifest = pd.DataFrame(records)
potato_manifest.to_csv(
    POTATO_EXPERIMENT_DIR / "potato_initial_manifest.csv",
    index=False,
)

print("Image files found:", len(image_paths))
print("Readable images:", len(potato_manifest))
print("Unreadable images:", len(unreadable))
print("\nImages per class:")
print(potato_manifest["label"].value_counts().sort_index())

if unreadable:
    unreadable_df = pd.DataFrame(unreadable)
    unreadable_df.to_csv(
        POTATO_EXPERIMENT_DIR / "potato_unreadable_images.csv",
        index=False,
    )
    print("\nUnreadable-image report saved.")
else:
    print("\nAll extracted images are readable.")

print("\nMost common image sizes:")
print(
    potato_manifest.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)
print("\nManifest saved to:", POTATO_EXPERIMENT_DIR / "potato_initial_manifest.csv")

Checking Potato images:   0%|          | 0/3076 [00:00<?, ?it/s]

Image files found: 3076
Readable images: 3076
Unreadable images: 0

Images per class:
label
Bacteria       569
Fungi          748
Healthy        201
Nematode        68
Pest           611
Phytopthora    347
Virus          532
Name: count, dtype: int64

All extracted images are readable.

Most common image sizes:
width  height
1500   1500      3076
dtype: int64

Manifest saved to: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_initial_manifest.csv


In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display
import pandas as pd
import random

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_initial_manifest.csv"

potato_df = pd.read_csv(MANIFEST_PATH)

# Keep Nematode in the original inventory, but exclude it from model training.
potato_df = potato_df[potato_df["label"] != "Nematode"].copy()
potato_df.to_csv(
    POTATO_EXPERIMENT_DIR / "potato_manifest_without_nematode.csv",
    index=False,
)

print("Nematode images excluded:", 68)
print("Images remaining:", len(potato_df))
print("\nRemaining images per class:")
print(potato_df["label"].value_counts().sort_index())

# Show up to two randomly selected images per class with their saved labels.
for label in sorted(potato_df["label"].unique()):
    samples = potato_df[potato_df["label"] == label].sample(
        n=min(2, (potato_df["label"] == label).sum()),
        random_state=42,
    )

    print(f"\n{label}")
    for _, row in samples.iterrows():
        print(Path(row["image_path"]).name)
        display(Image.open(row["image_path"]).convert("RGB"))

print("\nFiltered manifest saved to:", POTATO_EXPERIMENT_DIR / "potato_manifest_without_nematode.csv")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd
from tqdm.auto import tqdm

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_manifest_without_nematode.csv"

potato_df = pd.read_csv(MANIFEST_PATH)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

hashes = []
for path in tqdm(potato_df["image_path"], desc="Checking exact duplicates"):
    hashes.append(sha256_file(path))

potato_df["sha256"] = hashes

duplicate_groups = []
for sha256, group in potato_df.groupby("sha256"):
    if len(group) > 1:
        duplicate_groups.append({
            "sha256": sha256,
            "copies": len(group),
            "labels": ", ".join(sorted(group["label"].unique())),
            "image_paths": " | ".join(group["image_path"].tolist()),
            "conflicting_labels": group["label"].nunique() > 1,
        })

duplicate_report = pd.DataFrame(duplicate_groups)
report_path = POTATO_EXPERIMENT_DIR / "potato_exact_duplicate_report.csv"
duplicate_report.to_csv(report_path, index=False)

print("Images checked:", len(potato_df))
print("Duplicate groups:", len(duplicate_report))

if len(duplicate_report):
    print("Groups with conflicting labels:", int(duplicate_report["conflicting_labels"].sum()))
    print("Groups with one consistent label:", int((~duplicate_report["conflicting_labels"]).sum()))
    display(duplicate_report.head(20))
else:
    print("No exact duplicate files found.")

print("\nReport saved to:", report_path)

Checking exact duplicates:   0%|          | 0/3008 [00:00<?, ?it/s]

Images checked: 3008
Duplicate groups: 54
Groups with conflicting labels: 5
Groups with one consistent label: 49


,sha256,copies,labels,image_paths,conflicting_labels
0,07365d6f547280c0b283eb1097f0503e8b2173516100c0...,2,Virus,/content/potato_raw_fast/Virus/potato_02927.jp...,False
1,0cd994f85c76c526e22d71dfd0ee2ab1ea74a31392dbf2...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_01...,False
2,0f4656f45581f3439cb4688e2245a5312ef524ab9103b5...,2,Fungi,/content/potato_raw_fast/Fungi/potato_02215.jp...,False
3,1bddf338c236d74b16b23f4c6a26e66e6cf9c21684e83d...,2,"Pest, Virus",/content/potato_raw_fast/Virus/potato_02915.jp...,True
4,1c072947c322103a1ce53205bd4302ac19f4da1b3b463e...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_00...,False
5,1dfe0c4c6c06ad69df1aa7a769354132f3a2596f9f11ee...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_01...,False
6,1fe9f99b074fb41fe9086cc4bb4407d977a59a71372f40...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_00...,False
7,2313e011e69171761864bcf1b20b540cfa58609ee01cdb...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_01...,False
8,26ee54363b751b024fb2c4c0068b06cce734e951b0227a...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_01...,False
9,2d504280c2212acb74790f80c836791cab75e52869a005...,2,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_00...,False



Report saved to: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_exact_duplicate_report.csv


In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
import pandas as pd

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
REPORT_PATH = POTATO_EXPERIMENT_DIR / "potato_exact_duplicate_report.csv"

duplicates = pd.read_csv(REPORT_PATH)

conflicts = duplicates[duplicates["conflicting_labels"].astype(str).str.lower() == "true"]

print("Exact duplicate groups:", len(duplicates))
print("Conflicting-label groups:", len(conflicts))
print("Consistent-label duplicate groups:", len(duplicates) - len(conflicts))

for group_number, (_, row) in enumerate(conflicts.iterrows(), start=1):
    print(f"\nConflicting duplicate group {group_number}")
    print("Labels:", row["labels"])

    for image_path in row["image_paths"].split(" | "):
        path = Path(image_path)
        print(f"{path.parent.name} — {path.name}")
        display(Image.open(path).convert("RGB"))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pathlib import Path
import pandas as pd

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_manifest_without_nematode.csv"
REPORT_PATH = POTATO_EXPERIMENT_DIR / "potato_exact_duplicate_report.csv"

potato_df = pd.read_csv(MANIFEST_PATH)
duplicate_report = pd.read_csv(REPORT_PATH)

remove_paths = set()
removal_log = []

for _, group in duplicate_report.iterrows():
    paths = group["image_paths"].split(" | ")
    labels = group["labels"].split(", ")

    if len(set(labels)) > 1:
        # Conflicting labels: remove every copy in the group.
        paths_to_remove = paths
        reason = "exact_duplicate_conflicting_labels"
    else:
        # Same label: retain the first copy and remove the remaining copies.
        paths_to_remove = paths[1:]
        reason = "exact_duplicate_extra_copy"

    for path in paths_to_remove:
        remove_paths.add(path)
        removal_log.append({
            "image_path": path,
            "reason": reason,
            "duplicate_group_labels": group["labels"],
            "sha256": group["sha256"],
        })

clean_df = potato_df[~potato_df["image_path"].isin(remove_paths)].copy()

clean_manifest_path = POTATO_EXPERIMENT_DIR / "potato_clean_manifest.csv"
removal_log_path = POTATO_EXPERIMENT_DIR / "potato_exact_duplicates_removed.csv"

clean_df.to_csv(clean_manifest_path, index=False)
pd.DataFrame(removal_log).to_csv(removal_log_path, index=False)

print("Images before exact-duplicate cleanup:", len(potato_df))
print("Images removed:", len(remove_paths))
print("Images remaining:", len(clean_df))
print("\nRemaining images per class:")
print(clean_df["label"].value_counts().sort_index())
print("\nClean manifest:", clean_manifest_path)
print("Removal log:", removal_log_path)

Images before exact-duplicate cleanup: 3008
Images removed: 59
Images remaining: 2949

Remaining images per class:
label
Bacteria       569
Fungi          740
Healthy        199
Pest           602
Phytopthora    309
Virus          530
Name: count, dtype: int64

Clean manifest: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_clean_manifest.csv
Removal log: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_exact_duplicates_removed.csv


In [ ]:
%pip -q install ImageHash

from pathlib import Path
from itertools import combinations
import pandas as pd
import imagehash
from PIL import Image, ImageOps
from tqdm.auto import tqdm

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_clean_manifest.csv"

# Raise this to find more possible matches, at the cost of more false positives.
MAX_HASH_DISTANCE = 12

potato_df = pd.read_csv(MANIFEST_PATH)

def orientation_robust_hashes(image_path):
    """Hash the image at four right-angle rotations to catch rotated copies."""
    with Image.open(image_path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        return [
            imagehash.phash(image.rotate(angle, expand=True), hash_size=8)
            for angle in (0, 90, 180, 270)
        ]

hash_rows = []
for row in tqdm(
    potato_df.itertuples(index=False),
    total=len(potato_df),
    desc="Hashing Potato images",
):
    try:
        hashes = orientation_robust_hashes(row.image_path)
        hash_rows.append({
            "image_path": row.image_path,
            "label": row.label,
            "hashes": hashes,
        })
    except Exception as error:
        print(f"Could not hash {row.image_path}: {error}")

candidate_pairs = []
for first, second in tqdm(
    combinations(hash_rows, 2),
    total=len(hash_rows) * (len(hash_rows) - 1) // 2,
    desc="Comparing image pairs",
):
    distance = min(
        hash_a - hash_b
        for hash_a in first["hashes"]
        for hash_b in second["hashes"]
    )

    if distance <= MAX_HASH_DISTANCE:
        candidate_pairs.append({
            "image_path_1": first["image_path"],
            "label_1": first["label"],
            "image_path_2": second["image_path"],
            "label_2": second["label"],
            "hamming_distance": int(distance),
            "different_labels": first["label"] != second["label"],
        })

pairs_df = pd.DataFrame(candidate_pairs)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values(
        ["hamming_distance", "different_labels"],
        ascending=[True, False],
    ).reset_index(drop=True)

output_path = POTATO_EXPERIMENT_DIR / "potato_near_duplicate_candidates.csv"
pairs_df.to_csv(output_path, index=False)

print("Images hashed:", len(hash_rows))
print(f"Candidate pairs at distance <= {MAX_HASH_DISTANCE}:", len(pairs_df))
if not pairs_df.empty:
    print("Pairs with different labels:", int(pairs_df["different_labels"].sum()))
    display(pairs_df.head(30))

print("\nCandidate list saved to:", output_path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 11.5 MB/s eta 0:00:00


Hashing Potato images:   0%|          | 0/2949 [00:00<?, ?it/s]

Comparing image pairs:   0%|          | 0/4346826 [00:00<?, ?it/s]

Images hashed: 2949
Candidate pairs at distance <= 12: 170
Pairs with different labels: 63


,image_path_1,label_1,image_path_2,label_2,hamming_distance,different_labels
0,/content/potato_raw_fast/Phytopthora/potato_00...,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_01...,Phytopthora,0,False
1,/content/potato_raw_fast/Phytopthora/potato_00...,Phytopthora,/content/potato_raw_fast/Phytopthora/potato_00...,Phytopthora,0,False
2,/content/potato_raw_fast/Pest/potato_00549.jpg,Pest,/content/potato_raw_fast/Pest/potato_00061.jpg,Pest,0,False
3,/content/potato_raw_fast/Fungi/potato_02524.jpg,Fungi,/content/potato_raw_fast/Fungi/potato_02508.jpg,Fungi,0,False
4,/content/potato_raw_fast/Fungi/potato_02384.jpg,Fungi,/content/potato_raw_fast/Fungi/potato_02364.jpg,Fungi,0,False
5,/content/potato_raw_fast/Virus/potato_02595.jpg,Virus,/content/potato_raw_fast/Virus/potato_03055.jpg,Virus,2,False
6,/content/potato_raw_fast/Virus/potato_03049.jpg,Virus,/content/potato_raw_fast/Virus/potato_03072.jpg,Virus,2,False
7,/content/potato_raw_fast/Virus/potato_03017.jpg,Virus,/content/potato_raw_fast/Virus/potato_03034.jpg,Virus,2,False
8,/content/potato_raw_fast/Virus/potato_02749.jpg,Virus,/content/potato_raw_fast/Virus/potato_02786.jpg,Virus,2,False
9,/content/potato_raw_fast/Virus/potato_02628.jpg,Virus,/content/potato_raw_fast/Virus/potato_02601.jpg,Virus,2,False



Candidate list saved to: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_near_duplicate_candidates.csv


In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
CANDIDATES_PATH = POTATO_EXPERIMENT_DIR / "potato_near_duplicate_candidates.csv"
CONTACT_SHEET_PATH = POTATO_EXPERIMENT_DIR / "potato_near_duplicate_review.png"

pairs = pd.read_csv(CANDIDATES_PATH).sort_values("hamming_distance").reset_index(drop=True)

THUMB_W = 260
THUMB_H = 260
TEXT_H = 58
PAIRS_PER_SHEET = 24

def make_contact_sheet(pair_rows, sheet_number):
    sheet = Image.new(
        "RGB",
        (THUMB_W * 2, (THUMB_H + TEXT_H) * len(pair_rows)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for row_number, pair in enumerate(pair_rows):
        top = row_number * (THUMB_H + TEXT_H)
        left_path = Path(pair["image_path_1"])
        right_path = Path(pair["image_path_2"])

        for column, path in enumerate((left_path, right_path)):
            with Image.open(path) as image:
                image = ImageOps.exif_transpose(image).convert("RGB")
                image.thumbnail((THUMB_W, THUMB_H))
                x = column * THUMB_W + (THUMB_W - image.width) // 2
                y = top + (THUMB_H - image.height) // 2
                sheet.paste(image, (x, y))

        draw.text(
            (4, top + THUMB_H + 3),
            f"Pair {sheet_number * PAIRS_PER_SHEET + row_number + 1} | "
            f"{pair['label_1']} vs {pair['label_2']} | "
            f"distance={pair['hamming_distance']}",
            fill="black",
        )
        draw.text(
            (4, top + THUMB_H + 22),
            f"1: {left_path.name}",
            fill="black",
        )
        draw.text(
            (4, top + THUMB_H + 39),
            f"2: {right_path.name}",
            fill="black",
        )

    return sheet

print("Candidate pairs:", len(pairs))
print("Reviewing all pairs in contact sheets.")

for start in range(0, len(pairs), PAIRS_PER_SHEET):
    batch = pairs.iloc[start:start + PAIRS_PER_SHEET].to_dict("records")
    sheet_number = start // PAIRS_PER_SHEET
    contact_sheet = make_contact_sheet(batch, sheet_number)

    if sheet_number == 0:
        contact_sheet.save(CONTACT_SHEET_PATH)

    print(f"\nContact sheet {sheet_number + 1} — pairs {start + 1} to {start + len(batch)}")
    display(contact_sheet)

print("\nFirst contact sheet saved to:", CONTACT_SHEET_PATH)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pathlib import Path
import pandas as pd

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_clean_manifest.csv"
CANDIDATES_PATH = POTATO_EXPERIMENT_DIR / "potato_near_duplicate_candidates.csv"

# Pairs are numbered from 1 in the contact sheets.
# You confirmed pairs 1–79 as duplicates except 75, 78, and 79.
EXCLUDED_PAIR_NUMBERS = {75, 78, 79}
confirmed_pair_numbers = [
    number for number in range(1, 80)
    if number not in EXCLUDED_PAIR_NUMBERS
]

potato_df = pd.read_csv(MANIFEST_PATH)
pairs = pd.read_csv(CANDIDATES_PATH).sort_values(
    "hamming_distance"
).reset_index(drop=True)

if len(pairs) < 79:
    raise ValueError(f"Only {len(pairs)} candidate pairs are available; expected at least 79.")

confirmed = pairs.iloc[[number - 1 for number in confirmed_pair_numbers]].copy()

remove_paths = set()
removal_rows = []

for _, pair in confirmed.iterrows():
    path_1 = pair["image_path_1"]
    path_2 = pair["image_path_2"]
    labels_match = pair["label_1"] == pair["label_2"]

    if labels_match:
        # Keep image 1 and set aside image 2.
        remove_paths.add(path_2)
        removal_rows.append({
            "image_path": path_2,
            "label": pair["label_2"],
            "paired_image_path": path_1,
            "paired_label": pair["label_1"],
            "hamming_distance": pair["hamming_distance"],
            "action": "set_aside_extra_copy",
        })
    else:
        # Conflicting labels: set aside both copies so neither label is trusted.
        remove_paths.update((path_1, path_2))
        removal_rows.extend([
            {
                "image_path": path_1,
                "label": pair["label_1"],
                "paired_image_path": path_2,
                "paired_label": pair["label_2"],
                "hamming_distance": pair["hamming_distance"],
                "action": "set_aside_conflicting_label_pair",
            },
            {
                "image_path": path_2,
                "label": pair["label_2"],
                "paired_image_path": path_1,
                "paired_label": pair["label_1"],
                "hamming_distance": pair["hamming_distance"],
                "action": "set_aside_conflicting_label_pair",
            },
        ])

missing_paths = remove_paths - set(potato_df["image_path"])
if missing_paths:
    raise ValueError(
        f"{len(missing_paths)} reviewed paths are absent from the manifest; "
        "no files have been written."
    )

cleaned = potato_df[~potato_df["image_path"].isin(remove_paths)].copy()

clean_manifest_path = POTATO_EXPERIMENT_DIR / "potato_manifest_after_near_duplicate_review.csv"
removal_log_path = POTATO_EXPERIMENT_DIR / "potato_confirmed_near_duplicates_removed.csv"

cleaned.to_csv(clean_manifest_path, index=False)
pd.DataFrame(removal_rows).to_csv(removal_log_path, index=False)

print("Reviewed candidate pairs:", 79)
print("Confirmed duplicate pairs:", len(confirmed))
print("Excluded as non-duplicates:", sorted(EXCLUDED_PAIR_NUMBERS))
print("Images set aside:", len(remove_paths))
print("Images remaining:", len(cleaned))
print("\nRemaining images per class:")
print(cleaned["label"].value_counts().sort_index())
print("\nUpdated manifest:", clean_manifest_path)
print("Removal log:", removal_log_path)

Reviewed candidate pairs: 79
Confirmed duplicate pairs: 76
Excluded as non-duplicates: [75, 78, 79]
Images set aside: 79
Images remaining: 2870

Remaining images per class:
label
Bacteria       550
Fungi          716
Healthy        197
Pest           589
Phytopthora    300
Virus          518
Name: count, dtype: int64

Updated manifest: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_manifest_after_near_duplicate_review.csv
Removal log: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_confirmed_near_duplicates_removed.csv


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
MANIFEST_PATH = POTATO_EXPERIMENT_DIR / "potato_manifest_after_near_duplicate_review.csv"

RANDOM_STATE = 42

potato_df = pd.read_csv(MANIFEST_PATH)

# First hold out 15% for testing; split the remaining 85% into train and validation.
train_val_df, test_df = train_test_split(
    potato_df,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=potato_df["label"],
)

# 15% of the complete dataset for validation.
val_fraction_of_remaining = 0.15 / 0.85
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_fraction_of_remaining,
    random_state=RANDOM_STATE,
    stratify=train_val_df["label"],
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "internal_test"

train_path = POTATO_EXPERIMENT_DIR / "train_manifest.csv"
val_path = POTATO_EXPERIMENT_DIR / "validation_manifest.csv"
test_path = POTATO_EXPERIMENT_DIR / "internal_test_manifest.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Images per split and class:")
display(pd.crosstab(split_df["split"], split_df["label"]))

print("\nSplit totals:")
for name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("internal_test", test_df),
]:
    print(f"{name}: {len(frame)} ({len(frame) / len(potato_df):.1%})")

print("\nSaved manifests:")
print(train_path)
print(val_path)
print(test_path)

Images per split and class:


label,Bacteria,Fungi,Healthy,Pest,Phytopthora,Virus
split,,,,,,
internal_test,83,107,30,88,45,78
train,385,501,138,412,210,362
validation,82,108,29,89,45,78



Split totals:
train: 2008 (70.0%)
validation: 431 (15.0%)
internal_test: 431 (15.0%)

Saved manifests:
/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/train_manifest.csv
/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/validation_manifest.csv
/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/internal_test_manifest.csv


In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
from tqdm.auto import tqdm
import json

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
CACHE_DIR = Path("/content/potato_cache_448")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MANIFESTS = {
    "train": POTATO_EXPERIMENT_DIR / "train_manifest.csv",
    "validation": POTATO_EXPERIMENT_DIR / "validation_manifest.csv",
    "internal_test": POTATO_EXPERIMENT_DIR / "internal_test_manifest.csv",
}

IMAGE_SIZE = 448
cache_rows = []

all_frames = []
for split_name, manifest_path in MANIFESTS.items():
    frame = pd.read_csv(manifest_path)
    frame["split"] = split_name
    all_frames.append(frame)

all_images = pd.concat(all_frames, ignore_index=True).drop_duplicates("image_path")

for row in tqdm(all_images.itertuples(index=False), total=len(all_images), desc="Caching resized Potato images"):
    source_path = Path(row.image_path)
    cache_path = CACHE_DIR / row.label / f"{source_path.stem}.jpg"
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if not cache_path.exists():
        with Image.open(source_path) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = ImageOps.pad(
                image,
                (IMAGE_SIZE, IMAGE_SIZE),
                method=Image.Resampling.LANCZOS,
                color=(127, 127, 127),
            )
            image.save(cache_path, format="JPEG", quality=95)

    cache_rows.append({
        "image_path": str(row.image_path),
        "cache_path": str(cache_path),
        "label": row.label,
        "split": row.split,
    })

cache_manifest_path = POTATO_EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"
pd.DataFrame(cache_rows).to_csv(cache_manifest_path, index=False)

metadata = {
    "image_size": IMAGE_SIZE,
    "cache_dir": str(CACHE_DIR),
    "image_count": len(cache_rows),
    "source_manifest_files": {key: str(value) for key, value in MANIFESTS.items()},
}
with open(POTATO_EXPERIMENT_DIR / "potato_cache_config.json", "w") as file:
    json.dump(metadata, file, indent=2)

print("Images cached:", len(cache_rows))
print("Cache folder:", CACHE_DIR)
print("Cache manifest:", cache_manifest_path)

Caching resized Potato images:   0%|          | 0/2870 [00:00<?, ?it/s]

Images cached: 2870
Cache folder: /content/potato_cache_448
Cache manifest: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_resized_cache_manifest.csv


In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import RandomErasing
from tqdm.auto import tqdm

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
CACHE_MANIFEST = POTATO_EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"

IMAGE_SIZE = 448
TILE_SIZE = 288
BATCH_SIZE = 4
NUM_WORKERS = 2
PIN_MEMORY = torch.cuda.is_available()

cache_df = pd.read_csv(CACHE_MANIFEST)
CLASS_NAMES = sorted(cache_df["label"].unique())
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {index: name for name, index in CLASS_TO_IDX.items()}

train_df = cache_df[cache_df["split"] == "train"].reset_index(drop=True)
val_df = cache_df[cache_df["split"] == "validation"].reset_index(drop=True)
test_df = cache_df[cache_df["split"] == "internal_test"].reset_index(drop=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# Four overlapping corner tiles; the full image is also supplied to the model.
TILE_STARTS = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

def make_training_views(image):
    if random.random() < 0.5:
        image = TF.hflip(image)
    if random.random() < 0.5:
        image = TF.vflip(image)
    if random.random() < 0.5:
        image = TF.rotate(
            image,
            angle=random.uniform(-15, 15),
            fill=(127, 127, 127),
        )
    if random.random() < 0.5:
        image = ImageEnhance.Brightness(image).enhance(random.uniform(0.85, 1.15))
    if random.random() < 0.5:
        image = ImageEnhance.Contrast(image).enhance(random.uniform(0.85, 1.15))
    if random.random() < 0.25:
        image = ImageEnhance.Color(image).enhance(random.uniform(0.9, 1.1))
    if random.random() < 0.05:
        image = image.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 0.4)))
    return image

def image_to_tensor(image):
    tensor = TF.to_tensor(image)
    return TF.normalize(tensor, MEAN, STD)

class PotatoDataset(Dataset):
    def __init__(self, dataframe, training=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(row["cache_path"]) as source:
            image = source.convert("RGB")

        if self.training:
            image = make_training_views(image)

        full_image = image_to_tensor(image)

        tiles = []
        for left, top in TILE_STARTS:
            tile = image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            tiles.append(image_to_tensor(tile))

        target = CLASS_TO_IDX[row["label"]]
        return full_image, torch.stack(tiles), torch.tensor(target, dtype=torch.long)

train_loader = DataLoader(
    PotatoDataset(train_df, training=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    PotatoDataset(val_df, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    PotatoDataset(test_df, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

full_batch, tile_batch, target_batch = next(iter(train_loader))

print("Classes:", CLASS_NAMES)
print("Train / validation / internal test:", len(train_df), len(val_df), len(test_df))
print("Full image batch:", tuple(full_batch.shape))
print("Tile batch:", tuple(tile_batch.shape))
print("Label batch:", tuple(target_batch.shape))
print("Example label:", IDX_TO_CLASS[int(target_batch[0])])
print("GPU available:", torch.cuda.is_available())

Classes: ['Bacteria', 'Fungi', 'Healthy', 'Pest', 'Phytopthora', 'Virus']
Train / validation / internal test: 2008 431 431
Full image batch: (4, 3, 448, 448)
Tile batch: (4, 4, 3, 288, 288)
Label batch: (4,)
Example label: Healthy
GPU available: True


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch

# Give each training image a weight inversely proportional to its class frequency.
train_labels = [CLASS_TO_IDX[label] for label in train_df["label"]]
class_counts = torch.bincount(
    torch.tensor(train_labels, dtype=torch.long),
    minlength=len(CLASS_NAMES),
).float()

class_weights = 1.0 / class_counts
sample_weights = torch.tensor(
    [class_weights[label].item() for label in train_labels],
    dtype=torch.double,
)

balanced_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_labels),
    replacement=True,
)

train_loader = DataLoader(
    PotatoDataset(train_df, training=True),
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
)

print("Training image counts:")
for class_name, count in zip(CLASS_NAMES, class_counts.tolist()):
    print(f"  {class_name}: {int(count)}")

print("\nWeighted sampling enabled.")
print("Each epoch draws", len(train_loader) * BATCH_SIZE, "sample slots (last batch may be smaller).")

Training image counts:
  Bacteria: 385
  Fungi: 501
  Healthy: 138
  Pest: 412
  Phytopthora: 210
  Virus: 362

Weighted sampling enabled.
Each epoch draws 2008 sample slots (last batch may be smaller).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        backbone = resnet50(weights=weights)

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)

        weights = torch.softmax(self.tile_score(tile_features).squeeze(-1), dim=1)
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )
        return self.classifier(combined)


if "CLASS_NAMES" not in globals():
    raise NameError("CLASS_NAMES is missing. Run the Potato data-loader setup cell first.")

model = PotatoResNet50Attention(
    num_classes=len(CLASS_NAMES),
    pretrained=True,
).to(DEVICE)

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
BEST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_best_model.pth"
LAST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_last_checkpoint.pth"
HISTORY_PATH = POTATO_EXPERIMENT_DIR / "potato_training_history.csv"

EPOCHS = 15
PATIENCE = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Potato model initialized.")
print("Classes:", CLASS_NAMES)
print("Device:", DEVICE)
print("AMP enabled:", USE_AMP)
print("Best checkpoint exists:", BEST_CHECKPOINT_PATH.exists())
print("Latest checkpoint exists:", LAST_CHECKPOINT_PATH.exists())

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]


Potato model initialized.
Classes: ['Bacteria', 'Fungi', 'Healthy', 'Pest', 'Phytopthora', 'Virus']
Device: cuda
AMP enabled: True
Best checkpoint exists: False
Latest checkpoint exists: False


In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
BEST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_best_model.pth"
LAST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_last_checkpoint.pth"
HISTORY_PATH = POTATO_EXPERIMENT_DIR / "potato_training_history.csv"

EPOCHS = 20
PATIENCE = 6
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

start_epoch = 1
best_val_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
history = []

if LAST_CHECKPOINT_PATH.exists():
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val_f1 = float(checkpoint.get("best_val_f1", -1.0))
    best_epoch = int(checkpoint.get("best_epoch", 0))
    epochs_without_improvement = int(
        checkpoint.get("epochs_without_improvement", 0)
    )
    history = checkpoint.get("history", [])
    print(f"Resuming from epoch {start_epoch}; best validation macro-F1={best_val_f1:.4f}")
else:
    print("Starting a fresh Potato training run.")

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0
    y_true, y_pred = [], []

    progress = tqdm(
        loader,
        desc="Training" if training else "Validation",
        leave=False,
    )

    for full_images, tile_images, targets in progress:
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full_images, tile_images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size

        predictions = logits.argmax(dim=1)
        y_true.extend(targets.detach().cpu().tolist())
        y_pred.extend(predictions.detach().cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": total_loss / max(total_items, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0,
        ),
    }

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]

    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={row['train_loss']:.4f}, "
        f"accuracy={row['train_accuracy']:.4f}, "
        f"macro-F1={row['train_macro_f1']:.4f} | "
        f"Val loss={row['val_loss']:.4f}, "
        f"accuracy={row['val_accuracy']:.4f}, "
        f"balanced accuracy={row['val_balanced_accuracy']:.4f}, "
        f"macro-F1={row['val_macro_f1']:.4f}"
    )

    improved = val_metrics["macro_f1"] > best_val_f1
    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "best_val_f1": best_val_f1,
                "metrics": row,
            },
            BEST_CHECKPOINT_PATH,
        )
        print("Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "class_names": CLASS_NAMES,
        "best_val_f1": best_val_f1,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
    }
    torch.save(checkpoint, LAST_CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print("Saved latest resumable checkpoint.")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without validation macro-F1 improvement.")
        break

print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_f1:.4f} at epoch {best_epoch}")
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Latest checkpoint:", LAST_CHECKPOINT_PATH)
print("Training history:", HISTORY_PATH)

Resuming from epoch 11; best validation macro-F1=0.8638


Training:   0%|          | 0/502 [00:00<?, ?it/s]

Validation:   0%|          | 0/108 [00:00<?, ?it/s]

Epoch 11/20 | Train loss=0.4181, accuracy=0.8785, macro-F1=0.8785 | Val loss=0.4628, accuracy=0.8770, balanced accuracy=0.8740, macro-F1=0.8639
Saved new best checkpoint.
Saved latest resumable checkpoint.


Training:   0%|          | 0/502 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/108 [00:00<?, ?it/s]

Epoch 19/20 | Train loss=0.1806, accuracy=0.9487, macro-F1=0.9480 | Val loss=0.5149, accuracy=0.8886, balanced accuracy=0.8786, macro-F1=0.8764
Saved latest resumable checkpoint.


Training:   0%|          | 0/502 [00:00<?, ?it/s]

Validation:   0%|          | 0/108 [00:00<?, ?it/s]

Epoch 20/20 | Train loss=0.1797, accuracy=0.9502, macro-F1=0.9493 | Val loss=0.5286, accuracy=0.8840, balanced accuracy=0.8757, macro-F1=0.8720
Saved latest resumable checkpoint.
Early stopping after 6 epochs without validation macro-F1 improvement.

Training finished.
Best validation macro-F1: 0.8791 at epoch 14
Best checkpoint: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_best_model.pth
Latest checkpoint: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_last_checkpoint.pth
Training history: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_training_history.csv


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
BEST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_best_model.pth"
LAST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_last_checkpoint.pth"

# Rebuild the exact model architecture used by the training cell.
class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes, pretrained=False):
        super().__init__()
        backbone = resnet50(
            weights=ResNet50_Weights.DEFAULT if pretrained else None
        )

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)

        weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        )
        return self.classifier(combined)


if "CLASS_NAMES" not in globals():
    raise NameError("Run the Potato data-loader cell first so CLASS_NAMES is defined.")

model = PotatoResNet50Attention(
    num_classes=len(CLASS_NAMES),
    pretrained=False,
).to(DEVICE)

if not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"No resumable checkpoint found at:\n{LAST_CHECKPOINT_PATH}\n"
        "If this file is missing, the interrupted run did not save a checkpoint."
    )

checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

print("Latest checkpoint loaded successfully.")
print("Saved after epoch:", checkpoint["epoch"])
print("Best validation macro-F1:", checkpoint.get("best_val_f1"))
print("Best epoch:", checkpoint.get("best_epoch"))
print("Classes in checkpoint:", checkpoint.get("class_names"))
print("Device:", DEVICE)


Latest checkpoint loaded successfully.
Saved after epoch: 2
Best validation macro-F1: 0.7046206866558326
Best epoch: 2
Classes in checkpoint: ['Bacteria', 'Fungi', 'Healthy', 'Pest', 'Phytopthora', 'Virus']
Device: cuda


In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
LAST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_last_checkpoint.pth"
BEST_CHECKPOINT_PATH = POTATO_EXPERIMENT_DIR / "potato_best_model.pth"

if not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Resumable checkpoint not found:\n{LAST_CHECKPOINT_PATH}\n"
        "The interrupted run may not have completed and saved its first epoch."
    )

checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

CLASS_NAMES = checkpoint.get(
    "class_names",
    ["Bacteria", "Fungi", "Healthy", "Pest", "Phytopthora", "Virus"],
)

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet50(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=size,
            mode="bilinear", align_corners=False
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)

        weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        return self.classifier(
            torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)
        )


model = PotatoResNet50Attention(num_classes=len(CLASS_NAMES)).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)

print("Checkpoint loaded and model rebuilt.")
print("Saved epoch:", checkpoint["epoch"])
print("Best validation macro-F1:", checkpoint.get("best_val_f1"))
print("Best epoch:", checkpoint.get("best_epoch"))
print("Classes:", CLASS_NAMES)
print("Device:", DEVICE)
print("Latest checkpoint:", LAST_CHECKPOINT_PATH)
print("Best checkpoint exists:", BEST_CHECKPOINT_PATH.exists())

Checkpoint loaded and model rebuilt.
Saved epoch: 2
Best validation macro-F1: 0.7046206866558326
Best epoch: 2
Classes: ['Bacteria', 'Fungi', 'Healthy', 'Pest', 'Phytopthora', 'Virus']
Device: cuda
Latest checkpoint: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_last_checkpoint.pth
Best checkpoint exists: True


In [ ]:
from pathlib import Path

POTATO_ROOT = Path("/content/drive/MyDrive/final_year_project/Potato")

print("Searching under:", POTATO_ROOT)
if not POTATO_ROOT.exists():
    raise FileNotFoundError(
        f"Potato project folder not found: {POTATO_ROOT}. "
        "Make sure Google Drive is mounted."
    )

checkpoint_files = sorted(
    path for path in POTATO_ROOT.rglob("*")
    if path.is_file()
    and (
        "checkpoint" in path.name.lower()
        or path.name.lower().endswith((".pth", ".pt"))
    )
)

if checkpoint_files:
    print(f"Found {len(checkpoint_files)} checkpoint/model file(s):")
    for path in checkpoint_files:
        print(f"{path}  ({path.stat().st_size / (1024**2):.1f} MB)")
else:
    print("No .pth or .pt checkpoint files found under the Potato project folder.")

print("\nExperiment folder contents:")
experiment_dir = POTATO_ROOT / "potato_model_experiment"
if experiment_dir.exists():
    for path in sorted(experiment_dir.iterdir()):
        print(path.name, f"({path.stat().st_size / (1024**2):.1f} MB)" if path.is_file() else "[folder]")

Searching under: /content/drive/MyDrive/final_year_project/Potato
No .pth or .pt checkpoint files found under the Potato project folder.

Experiment folder contents:


In [ ]:
from pathlib import Path

POTATO_EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
CACHE_DIR = Path("/content/potato_cache_448")

print("Drive experiment folder exists:", POTATO_EXPERIMENT_DIR.exists())
print("\nSaved files in experiment folder:")
if POTATO_EXPERIMENT_DIR.exists():
    for path in sorted(POTATO_EXPERIMENT_DIR.iterdir()):
        kind = "folder" if path.is_dir() else f"{path.stat().st_size / (1024**2):.2f} MB"
        print(f"  {path.name}: {kind}")

print("\nColab cache exists:", CACHE_DIR.exists())
if CACHE_DIR.exists():
    cached_images = list(CACHE_DIR.rglob("*.jpg"))
    print("Cached images:", len(cached_images))
else:
    print("No local image cache found.")

for filename in [
    "train_manifest.csv",
    "validation_manifest.csv",
    "internal_test_manifest.csv",
    "potato_resized_cache_manifest.csv",
]:
    path = POTATO_EXPERIMENT_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

Drive experiment folder exists: False

Saved files in experiment folder:

Colab cache exists: False
No local image cache found.
train_manifest.csv: MISSING
validation_manifest.csv: MISSING
internal_test_manifest.csv: MISSING
potato_resized_cache_manifest.csv: MISSING


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

POTATO_ROOT = Path("/content/drive/MyDrive/final_year_project/Potato")
ZIP_PATH = POTATO_ROOT / "Potato Leaf Disease Image.zip"

print("Potato project folder exists:", POTATO_ROOT.exists())
print("Source ZIP exists:", ZIP_PATH.exists())

if POTATO_ROOT.exists():
    print("\nItems in Potato project folder:")
    for path in sorted(POTATO_ROOT.iterdir()):
        print(path.name, "[folder]" if path.is_dir() else f"({path.stat().st_size / (1024**2):.1f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Potato project folder exists: True
Source ZIP exists: False

Items in Potato project folder:
checkpoints [folder]


In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
matches = [
    path for path in drive_root.rglob("*")
    if path.is_file()
    and path.suffix.lower() == ".zip"
    and "potato" in path.name.lower()
]

if matches:
    print("Potato ZIP file(s) found:")
    for path in matches:
        print(path)
else:
    print("No ZIP with 'potato' in its filename found in MyDrive.")

No ZIP with 'potato' in its filename found in MyDrive.


In [ ]:
from pathlib import Path

required = [
    ("training loader", "train_loader"),
    ("validation loader", "val_loader"),
    ("model", "model"),
    ("optimizer", "optimizer"),
    ("scheduler", "scheduler"),
    ("gradient scaler", "scaler"),
    ("training dataframe", "train_df"),
    ("validation dataframe", "val_df"),
    ("class names", "CLASS_NAMES"),
]

print("Runtime state:")
missing = []
for label, name in required:
    exists = name in globals()
    print(f"{label}: {'ready' if exists else 'missing'}")
    if not exists:
        missing.append(name)

cache_dir = Path("/content/potato_cache_448")
print("\nLocal image cache exists:", cache_dir.exists())
if cache_dir.exists():
    print("Cached JPG images:", len(list(cache_dir.rglob("*.jpg"))))

print("\nCheckpoint exists:", LAST_CHECKPOINT_PATH.exists())
print("Checkpoint path:", LAST_CHECKPOINT_PATH)

if missing:
    print("\nMissing runtime variables:", ", ".join(missing))
    print("The runtime needs the data-loader setup restored before training can resume.")
else:
    print("\nRuntime is ready to resume training from the saved checkpoint.")

Runtime state:
training loader: ready
validation loader: ready
model: ready
optimizer: ready
scheduler: ready
gradient scaler: ready
training dataframe: ready
validation dataframe: ready
class names: ready

Local image cache exists: True
Cached JPG images: 2870

Checkpoint exists: True
Checkpoint path: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_last_checkpoint.pth

Runtime is ready to resume training from the saved checkpoint.


In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

# Keep the same training settings used for this run.
EPOCHS = 20
PATIENCE = 8
GRAD_CLIP_NORM = 1.0

checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
scaler.load_state_dict(checkpoint["scaler_state_dict"])

start_epoch = int(checkpoint["epoch"]) + 1
best_val_f1 = float(checkpoint.get("best_val_f1", -1.0))
best_epoch = int(checkpoint.get("best_epoch", 0))
epochs_without_improvement = int(
    checkpoint.get("epochs_without_improvement", 0)
)
history = checkpoint.get("history", [])

print(f"Resuming at epoch {start_epoch}")
print(f"Best validation macro-F1 so far: {best_val_f1:.4f} (epoch {best_epoch})")

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0
    y_true, y_pred = [], []

    progress = tqdm(
        loader,
        desc="Training" if training else "Validation",
        leave=False,
    )

    for full_images, tile_images, targets in progress:
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full_images, tile_images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRAD_CLIP_NORM
                )
                scaler.step(optimizer)
                scaler.update()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size
        y_true.extend(targets.detach().cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).detach().cpu().tolist())

        progress.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": total_loss / max(total_items, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0,
        ),
    }

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]

    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={row['train_loss']:.4f}, "
        f"macro-F1={row['train_macro_f1']:.4f} | "
        f"Val loss={row['val_loss']:.4f}, "
        f"accuracy={row['val_accuracy']:.4f}, "
        f"balanced accuracy={row['val_balanced_accuracy']:.4f}, "
        f"macro-F1={row['val_macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "best_val_f1": best_val_f1,
                "metrics": row,
            },
            BEST_CHECKPOINT_PATH,
        )
        print("Saved new best checkpoint.")
    else:
        epochs_without_improvement += 1

    # Save full optimizer/scheduler/scaler state after every completed epoch.
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "class_names": CLASS_NAMES,
            "best_val_f1": best_val_f1,
            "best_epoch": best_epoch,
            "epochs_without_improvement": epochs_without_improvement,
            "history": history,
        },
        LAST_CHECKPOINT_PATH,
    )
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print("Saved resumable checkpoint to Drive.")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no validation macro-F1 improvement for {PATIENCE} epochs.")
        break

print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_f1:.4f} at epoch {best_epoch}")
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Latest checkpoint:", LAST_CHECKPOINT_PATH)

Resuming at epoch 3
Best validation macro-F1 so far: 0.7046 (epoch 2)


Training:   0%|          | 0/502 [00:00<?, ?it/s]

AssertionError: No inf checks were recorded for this optimizer.

In [ ]:
from torch.utils.data import DataLoader

# Recreate the loaders without background workers.
# Keep the weighted sampler for training so the smaller classes are sampled more often.
train_loader = DataLoader(
    PotatoDataset(train_df, training=True),
    batch_size=BATCH_SIZE,
    sampler=balanced_sampler,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=False,
)

val_loader = DataLoader(
    PotatoDataset(val_df, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=False,
)

test_loader = DataLoader(
    PotatoDataset(test_df, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=False,
)

# Confirm a batch can be read before resuming training.
full_batch, tile_batch, target_batch = next(iter(train_loader))

print("Training loader ready:", len(train_loader), "batches")
print("Validation loader ready:", len(val_loader), "batches")
print("Batch shapes:", tuple(full_batch.shape), tuple(tile_batch.shape), tuple(target_batch.shape))
print("Resume checkpoint exists:", LAST_CHECKPOINT_PATH.exists())
print("Next training epoch:", torch.load(
    LAST_CHECKPOINT_PATH, map_location="cpu", weights_only=False
)["epoch"] + 1)

Training loader ready: 502 batches
Validation loader ready: 108 batches
Batch shapes: (4, 3, 448, 448) (4, 4, 3, 288, 288) (4,)
Resume checkpoint exists: True
Next training epoch: 3


In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

EPOCHS = 20
PATIENCE = 8
GRAD_CLIP_NORM = 1.0

# Always restore the last fully completed epoch before resuming.
checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

# Start a fresh scaler so no interrupted-batch AMP state carries over.
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
optimizer.zero_grad(set_to_none=True)

start_epoch = int(checkpoint["epoch"]) + 1
best_val_f1 = float(checkpoint.get("best_val_f1", -1.0))
best_epoch = int(checkpoint.get("best_epoch", 0))
epochs_without_improvement = int(checkpoint.get("epochs_without_improvement", 0))
history = checkpoint.get("history", [])

print(f"Resuming at epoch {start_epoch}")
print(f"Best validation macro-F1 so far: {best_val_f1:.4f} (epoch {best_epoch})")

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_items = 0
    y_true, y_pred = [], []

    for full_images, tile_images, targets in tqdm(
        loader,
        desc="Training" if training else "Validation",
        leave=False,
    ):
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(full_images, tile_images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), GRAD_CLIP_NORM
                )
                scaler.step(optimizer)
                scaler.update()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_items += batch_size
        y_true.extend(targets.detach().cpu().tolist())
        y_pred.extend(logits.argmax(dim=1).detach().cpu().tolist())

    return {
        "loss": total_loss / max(total_items, 1),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0,
        ),
    }

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]["lr"]
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics["macro_f1"])

    row = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_balanced_accuracy": train_metrics["balanced_accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_balanced_accuracy": val_metrics["balanced_accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss={row['train_loss']:.4f}, macro-F1={row['train_macro_f1']:.4f} | "
        f"Val loss={row['val_loss']:.4f}, accuracy={row['val_accuracy']:.4f}, "
        f"balanced accuracy={row['val_balanced_accuracy']:.4f}, "
        f"macro-F1={row['val_macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "best_val_f1": best_val_f1,
                "metrics": row,
            },
            BEST_CHECKPOINT_PATH,
        )
        print("Saved new best model checkpoint.")

    else:
        epochs_without_improvement += 1

    # Save after every fully completed epoch. A later interruption resumes here.
    latest_state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "class_names": CLASS_NAMES,
        "best_val_f1": best_val_f1,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
    }
    torch.save(latest_state, LAST_CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print("Saved completed-epoch checkpoint to Drive.")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break

print("\nTraining finished.")
print(f"Best validation macro-F1: {best_val_f1:.4f} at epoch {best_epoch}")
print("Best model:", BEST_CHECKPOINT_PATH)
print("Latest resumable checkpoint:", LAST_CHECKPOINT_PATH)

Resuming at epoch 3
Best validation macro-F1 so far: 0.7046 (epoch 2)


Training:   0%|          | 0/502 [00:00<?, ?it/s]

AssertionError: No inf checks were recorded for this optimizer.

In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Load the checkpoint selected by validation macro-F1.
best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
model.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for full_images, tile_images, targets in tqdm(
        test_loader,
        desc="Evaluating Potato internal test set",
    ):
        full_images = full_images.to(DEVICE, non_blocking=True)
        tile_images = tile_images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = model(full_images, tile_images)

        y_true.extend(targets.tolist())
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())

test_accuracy = accuracy_score(y_true, y_pred)
test_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    average="macro",
    zero_division=0,
)

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
)

print("Best checkpoint epoch:", best_checkpoint["epoch"])
print(f"Internal test images: {len(y_true)}")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Balanced accuracy: {test_balanced_accuracy:.4f}")
print(f"Macro-F1: {test_macro_f1:.4f}")
print("\nPer-class metrics:")
display(pd.DataFrame(report).T.round(4))
print("\nConfusion matrix (true rows, predicted columns):")
display(pd.DataFrame(confusion, index=CLASS_NAMES, columns=CLASS_NAMES))

# Save the evaluation outputs to Drive before stopping the runtime.
POTATO_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(report).T.to_csv(
    POTATO_EXPERIMENT_DIR / "potato_internal_test_classification_report.csv"
)
pd.DataFrame(
    confusion,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
).to_csv(POTATO_EXPERIMENT_DIR / "potato_internal_test_confusion_matrix.csv")

pd.DataFrame({
    "true_index": y_true,
    "predicted_index": y_pred,
    "true_label": [CLASS_NAMES[index] for index in y_true],
    "predicted_label": [CLASS_NAMES[index] for index in y_pred],
}).to_csv(
    POTATO_EXPERIMENT_DIR / "potato_internal_test_predictions.csv",
    index=False,
)

print("\nEvaluation reports saved in:", POTATO_EXPERIMENT_DIR)

Evaluating Potato internal test set:   0%|          | 0/108 [00:00<?, ?it/s]

Best checkpoint epoch: 14
Internal test images: 431
Accuracy: 0.8886
Balanced accuracy: 0.8896
Macro-F1: 0.8807

Per-class metrics:


,precision,recall,f1-score,support
Bacteria,0.9625,0.9277,0.9448,83.0000
Fungi,0.8390,0.9252,0.8800,107.0000
Healthy,0.7143,1.0000,0.8333,30.0000
Pest,0.8764,0.8864,0.8814,88.0000
Phytopthora,0.9211,0.7778,0.8434,45.0000
Virus,1.0000,0.8205,0.9014,78.0000
accuracy,0.8886,0.8886,0.8886,0.8886
macro avg,0.8855,0.8896,0.8807,431.0000
weighted avg,0.8994,0.8886,0.8896,431.0000



Confusion matrix (true rows, predicted columns):


,Bacteria,Fungi,Healthy,Pest,Phytopthora,Virus
Bacteria,77,3,0,3,0,0
Fungi,1,99,0,4,3,0
Healthy,0,0,30,0,0,0
Pest,0,4,6,78,0,0
Phytopthora,0,8,0,2,35,0
Virus,2,4,6,2,0,64



Evaluation reports saved in: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment


In [1]:
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)

print("Experiment folder exists:", EXPERIMENT_DIR.exists())

if EXPERIMENT_DIR.exists():
    print("\nSaved experiment files:")
    for path in sorted(EXPERIMENT_DIR.iterdir()):
        if path.is_file():
            print(f"{path.name} — {path.stat().st_size / (1024 * 1024):.2f} MB")
        else:
            print(f"{path.name}/")

print("\nTraining history exists:",
      (EXPERIMENT_DIR / "potato_training_history.csv").exists())
print("Validation manifest exists:",
      (EXPERIMENT_DIR / "validation_manifest.csv").exists())
print("Best checkpoint exists:",
      (EXPERIMENT_DIR / "potato_best_model.pth").exists())

cache_dir = Path("/content/potato_cache_448")
print("\nColab image cache exists:", cache_dir.exists())
if cache_dir.exists():
    print("Cached images:", len(list(cache_dir.rglob("*.jpg"))))

Experiment folder exists: True

Saved experiment files:
internal_test_manifest.csv — 0.03 MB
potato_best_model.pth — 103.28 MB
potato_cache_config.json — 0.00 MB
potato_clean_manifest.csv — 0.19 MB
potato_confirmed_near_duplicates_removed.csv — 0.01 MB
potato_exact_duplicate_report.csv — 0.01 MB
potato_exact_duplicates_removed.csv — 0.01 MB
potato_initial_manifest.csv — 0.19 MB
potato_internal_test_classification_report.csv — 0.00 MB
potato_internal_test_confusion_matrix.csv — 0.00 MB
potato_internal_test_predictions.csv — 0.01 MB
potato_last_checkpoint.pth — 309.36 MB
potato_manifest_after_near_duplicate_review.csv — 0.18 MB
potato_manifest_without_nematode.csv — 0.19 MB
potato_near_duplicate_candidates.csv — 0.02 MB
potato_near_duplicate_review.png — 5.29 MB
potato_resized_cache_manifest.csv — 0.31 MB
potato_training_history.csv — 0.00 MB
train_manifest.csv — 0.14 MB
validation_manifest.csv — 0.03 MB

Training history exists: True
Validation manifest exists: True
Best checkpoint exis

In [3]:
from pathlib import Path
import pandas as pd
import torch

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
HISTORY_PATH = EXPERIMENT_DIR / "potato_training_history.csv"
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "potato_best_model.pth"

history = pd.read_csv(HISTORY_PATH)
checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Training history by epoch:")
display(
    history[
        [
            "epoch",
            "train_loss",
            "train_macro_f1",
            "val_loss",
            "val_accuracy",
            "val_balanced_accuracy",
            "val_macro_f1",
        ]
    ].round(4)
)

print("\nBest checkpoint epoch:", checkpoint["epoch"])
print("Best validation macro-F1:", checkpoint["best_val_f1"])
print("Best checkpoint validation metrics:")
print(checkpoint.get("metrics", "No metrics stored in checkpoint"))

print("\nValidation manifest:", (EXPERIMENT_DIR / "validation_manifest.csv").exists())
print("Training manifest:", (EXPERIMENT_DIR / "train_manifest.csv").exists())

Training history by epoch:


,epoch,train_loss,train_macro_f1,val_loss,val_accuracy,val_balanced_accuracy,val_macro_f1
0,1,1.3847,0.4669,1.0343,0.6543,0.6945,0.6385
1,2,0.9678,0.6496,0.8053,0.7216,0.7559,0.7046
2,3,0.7672,0.7332,0.6344,0.7610,0.7925,0.7557
3,4,0.6918,0.7564,0.5547,0.8260,0.8356,0.8123
4,5,0.6141,0.7851,0.5326,0.8167,0.8329,0.7953
5,6,0.5498,0.8110,0.4456,0.8492,0.8602,0.8417
6,7,0.4977,0.8401,0.4094,0.8538,0.8468,0.8354
7,8,0.4884,0.8406,0.4133,0.8724,0.8672,0.8586
8,9,0.4175,0.8630,0.4601,0.8631,0.8652,0.8539
9,10,0.4382,0.8712,0.3927,0.8724,0.8666,0.8638



Best checkpoint epoch: 14
Best validation macro-F1: 0.8790890892427757
Best checkpoint validation metrics:
{'epoch': 14, 'learning_rate': 1e-05, 'train_loss': 0.3046337969868307, 'train_accuracy': 0.9083665338645418, 'train_balanced_accuracy': np.float64(0.9096512276544052), 'train_macro_f1': 0.9090091257988849, 'val_loss': 0.45227523303613404, 'val_accuracy': 0.8909512761020881, 'val_balanced_accuracy': np.float64(0.8889694290416578), 'val_macro_f1': 0.8790890892427757}

Validation manifest: True
Training manifest: True


In [4]:
%pip -q install datasets

from pathlib import Path
from datasets import load_dataset, Image as HFImage
from PIL import Image, ImageOps
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tqdm.auto import tqdm
import pandas as pd
import torch
import io
import numpy as np

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
CACHE_MANIFEST_PATH = EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"
VALIDATION_PATH = EXPERIMENT_DIR / "validation_manifest.csv"
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "potato_best_model.pth"
CACHE_DIR = Path("/content/potato_cache_448")
IMAGE_SIZE = 448

# Recreate the temporary cache directly from the Hugging Face dataset.
if not CACHE_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing cache manifest: {CACHE_MANIFEST_PATH}")

cache_manifest = pd.read_csv(CACHE_MANIFEST_PATH)
missing_cache = cache_manifest[
    ~cache_manifest["cache_path"].map(lambda value: Path(value).exists())
].copy()

if len(missing_cache):
    print("Reloading Potato image bytes from Hugging Face...")
    potato_source = load_dataset(
        "Project-AgML/potato_leaf_disease_classification",
        split="train",
    ).cast_column("image", HFImage(decode=False))

    # Filenames potato_00000.jpg ... correspond to the source dataset row order.
    source_index_by_name = {
        f"potato_{index:05d}": index
        for index in range(len(potato_source))
    }

    for row in tqdm(missing_cache.itertuples(index=False), total=len(missing_cache), desc="Restoring validation cache"):
        cache_path = Path(row.cache_path)
        source_path = Path(row.image_path)
        source_index = source_index_by_name.get(source_path.stem)

        if source_index is None:
            raise ValueError(f"Cannot map source image back to HF row: {source_path.name}")

        image_info = potato_source[source_index]["image"]
        raw_bytes = image_info.get("bytes")
        if raw_bytes is None:
            hf_path = image_info.get("path")
            if not hf_path:
                raise ValueError(f"No image bytes/path for source row {source_index}")
            raw_bytes = Path(hf_path).read_bytes()

        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(io.BytesIO(raw_bytes)) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = ImageOps.pad(
                image,
                (IMAGE_SIZE, IMAGE_SIZE),
                method=Image.Resampling.LANCZOS,
                color=(127, 127, 127),
            )
            image.save(cache_path, format="JPEG", quality=95)

print("Missing cache files after restore:",
      sum(not Path(path).exists() for path in cache_manifest["cache_path"]))

# Set up validation-only dataset with the same deterministic full-image/tile preprocessing.
checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
CLASS_NAMES = checkpoint["class_names"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for name, idx in CLASS_TO_IDX.items()}

validation_df = pd.read_csv(VALIDATION_PATH)
cache_lookup = cache_manifest.set_index("image_path")["cache_path"].to_dict()
validation_df["cache_path"] = validation_df["image_path"].map(cache_lookup)

if validation_df["cache_path"].isna().any():
    raise ValueError("Some validation images are missing from the cache manifest.")
if not validation_df["cache_path"].map(lambda value: Path(value).exists()).all():
    raise FileNotFoundError("Some validation cache images are still missing.")

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
TILE_SIZE = 288
TILE_STARTS = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

def normalized_tensor(image):
    return TF.normalize(TF.to_tensor(image), MEAN, STD)

class PotatoValidationDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["cache_path"]) as source:
            image = source.convert("RGB")

        full = normalized_tensor(image)
        tiles = [
            normalized_tensor(
                image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE))
            )
            for left, top in TILE_STARTS
        ]
        return full, torch.stack(tiles), CLASS_TO_IDX[row["label"]]

validation_loader = DataLoader(
    PotatoValidationDataset(validation_df),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

# Rebuild the same architecture if this runtime does not already have the model.
if "model" not in globals():
    raise NameError(
        "The Potato model class is not defined in this runtime. "
        "Run the existing PotatoResNet50Attention definition cell first."
    )

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE).eval()
USE_AMP = DEVICE.type == "cuda"

y_true, y_pred, confidence = [], [], []
with torch.inference_mode():
    for full, tiles, targets in tqdm(validation_loader, desc="Evaluating validation set"):
        full = full.to(DEVICE, non_blocking=True)
        tiles = tiles.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = model(full, tiles)

        probabilities = torch.softmax(logits.float(), dim=1)
        scores, predictions = probabilities.max(dim=1)

        y_true.extend(targets.tolist())
        y_pred.extend(predictions.cpu().tolist())
        confidence.extend(scores.cpu().tolist())

print("\nValidation macro-F1:",
      f1_score(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), average="macro", zero_division=0))
print("\nPer-class validation report:")
print(classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES, zero_division=0,
))
print("Confusion matrix (true rows, predicted columns):")
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES)))),
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
))

validation_results = validation_df[["image_path", "label"]].copy()
validation_results["true_index"] = y_true
validation_results["predicted_index"] = y_pred
validation_results["predicted_label"] = [IDX_TO_CLASS[index] for index in y_pred]
validation_results["confidence"] = confidence
validation_results["correct"] = validation_results["label"] == validation_results["predicted_label"]

results_path = EXPERIMENT_DIR / "potato_validation_predictions_epoch14.csv"
confusion_path = EXPERIMENT_DIR / "potato_validation_confusion_epoch14.csv"
validation_results.to_csv(results_path, index=False)
pd.crosstab(
    validation_results["label"],
    validation_results["predicted_label"],
    rownames=["true_label"],
    colnames=["predicted_label"],
).to_csv(confusion_path)

print("\nValidation predictions saved:", results_path)
print("Validation confusion matrix saved:", confusion_path)

Reloading Potato image bytes from Hugging Face...


README.md:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  381MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  373MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3076 [00:00<?, ? examples/s]

Restoring validation cache:   0%|          | 0/2870 [00:00<?, ?it/s]

Missing cache files after restore: 0


NameError: The Potato model class is not defined in this runtime. Run the existing PotatoResNet50Attention definition cell first.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
)

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "potato_best_model.pth"
VALIDATION_PATH = EXPERIMENT_DIR / "validation_manifest.csv"
CACHE_MANIFEST_PATH = EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"
IMAGE_SIZE, TILE_SIZE, BATCH_SIZE = 448, 288, 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
CLASS_NAMES = checkpoint["class_names"]
CLASS_TO_IDX = {name: index for index, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {index: name for name, index in CLASS_TO_IDX.items()}

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        attended, _ = self.attention(
            self.norm(tokens), self.norm(tokens), self.norm(tokens),
            need_weights=False,
        )
        attended = attended.transpose(1, 2).reshape(batch, channels, height, width)
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended

class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        from torchvision.models import resnet50
        backbone = resnet50(weights=None)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1, self.layer2 = backbone.layer1, backbone.layer2
        self.layer3, self.layer4 = backbone.layer3, backbone.layer4
        self.project2 = nn.Conv2d(512, 256, 1)
        self.project3 = nn.Conv2d(1024, 256, 1)
        self.project4 = nn.Conv2d(2048, 256, 1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle, deep = self.layer3(fine), None
        deep = self.layer4(middle)
        size = fine.shape[-2:]
        fused = self.fusion(torch.cat([
            self.project2(fine),
            F.interpolate(self.project3(middle), size=size, mode="bilinear", align_corners=False),
            F.interpolate(self.project4(deep), size=size, mode="bilinear", align_corners=False),
        ], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)
        weights = torch.softmax(self.tile_score(tile_features).squeeze(-1), dim=1)
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        return self.classifier(torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        ))

model = PotatoResNet50Attention(num_classes=len(CLASS_NAMES)).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

cache_manifest = pd.read_csv(CACHE_MANIFEST_PATH)
cache_lookup = dict(zip(cache_manifest["image_path"], cache_manifest["cache_path"]))
validation_df = pd.read_csv(VALIDATION_PATH)
validation_df["cache_path"] = validation_df["image_path"].map(cache_lookup)

if validation_df["cache_path"].isna().any():
    raise ValueError("Some validation images are missing from the cache manifest.")
missing = [p for p in validation_df["cache_path"] if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} validation cache files are missing.")

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
TILE_STARTS = [
    (0, 0),
    (IMAGE_SIZE - TILE_SIZE, 0),
    (0, IMAGE_SIZE - TILE_SIZE),
    (IMAGE_SIZE - TILE_SIZE, IMAGE_SIZE - TILE_SIZE),
]

def make_tensor(image):
    return TF.normalize(TF.to_tensor(image), MEAN, STD)

class PotatoValidationDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["cache_path"]) as source:
            image = source.convert("RGB")
        full = make_tensor(image)
        tiles = torch.stack([
            make_tensor(image.crop((left, top, left + TILE_SIZE, top + TILE_SIZE)))
            for left, top in TILE_STARTS
        ])
        return full, tiles, CLASS_TO_IDX[row["label"]]

val_loader = DataLoader(
    PotatoValidationDataset(validation_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=USE_AMP,
)

y_true, y_pred, confidences = [], [], []
with torch.inference_mode():
    for full, tiles, targets in tqdm(val_loader, desc="Validation-only evaluation"):
        full, tiles = full.to(DEVICE), tiles.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(full, tiles)
        probs = torch.softmax(logits.float(), dim=1)
        confidence, prediction = probs.max(dim=1)
        y_true.extend(targets.tolist())
        y_pred.extend(prediction.cpu().tolist())
        confidences.extend(confidence.cpu().tolist())

print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Validation balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
print("Validation macro-F1:", round(f1_score(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
    average="macro", zero_division=0
), 4))
print("\nPer-class report:")
print(classification_report(
    y_true, y_pred, labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES, zero_division=0,
))
print("Confusion matrix (true rows, predicted columns):")
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES)))),
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
))

results = validation_df[["image_path", "label"]].copy()
results["predicted_label"] = [IDX_TO_CLASS[i] for i in y_pred]
results["confidence"] = confidences
results["correct"] = results["label"] == results["predicted_label"]
results_path = EXPERIMENT_DIR / "potato_validation_predictions_epoch14.csv"
results.to_csv(results_path, index=False)

print("Validation predictions saved:", results_path)

Validation-only evaluation:   0%|          | 0/108 [00:00<?, ?it/s]

Checkpoint epoch: 14
Validation accuracy: 0.891
Validation balanced accuracy: 0.889
Validation macro-F1: 0.8791

Per-class report:
              precision    recall  f1-score   support

    Bacteria       0.98      0.96      0.97        82
       Fungi       0.88      0.91      0.89       108
     Healthy       0.66      0.93      0.77        29
        Pest       0.87      0.88      0.87        89
 Phytopthora       0.95      0.82      0.88        45
       Virus       0.96      0.83      0.89        78

    accuracy                           0.89       431
   macro avg       0.88      0.89      0.88       431
weighted avg       0.90      0.89      0.89       431

Confusion matrix (true rows, predicted columns):


,Bacteria,Fungi,Healthy,Pest,Phytopthora,Virus
Bacteria,79,1,0,2,0,0
Fungi,0,98,2,6,1,1
Healthy,0,0,27,1,0,1
Pest,0,5,5,78,0,1
Phytopthora,1,6,0,1,37,0
Virus,1,2,7,2,1,65


Validation predictions saved: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/potato_validation_predictions_epoch14.csv


In [6]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
PREDICTIONS_PATH = EXPERIMENT_DIR / "potato_validation_predictions_epoch14.csv"
REVIEW_PATH = EXPERIMENT_DIR / "potato_validation_targeted_error_review.csv"

predictions = pd.read_csv(PREDICTIONS_PATH)

# Prioritize the observed validation confusions:
# Healthy -> any disease, Phytopthora -> Fungi, Virus -> Healthy/Fungi.
targeted = predictions[
    (~predictions["correct"])
    & (
        (predictions["label"] == "Healthy")
        | (
            (predictions["label"] == "Phytopthora")
            & (predictions["predicted_label"] == "Fungi")
        )
        | (
            (predictions["label"] == "Virus")
            & (predictions["predicted_label"].isin(["Healthy", "Fungi"]))
        )
    )
].copy()

targeted = targeted.sort_values("confidence", ascending=False).reset_index(drop=True)
targeted.to_csv(REVIEW_PATH, index=False)

print("Targeted validation errors:", len(targeted))
print(targeted[["label", "predicted_label", "confidence", "image_path"]].to_string(index=False))

# Display readable image cards for manual checking.
MAX_IMAGES_TO_SHOW = min(40, len(targeted))
CARD_W, CARD_H, TEXT_H = 320, 320, 58
cards = []

for _, row in targeted.head(MAX_IMAGES_TO_SHOW).iterrows():
    path = Path(row["image_path"])
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        image.thumbnail((CARD_W, CARD_H))

    card = Image.new("RGB", (CARD_W, CARD_H + TEXT_H), "white")
    x = (CARD_W - image.width) // 2
    y = (CARD_H - image.height) // 2
    card.paste(image, (x, y))

    draw = ImageDraw.Draw(card)
    draw.text((5, CARD_H + 4), f"True: {row['label']} | Pred: {row['predicted_label']}", fill="black")
    draw.text((5, CARD_H + 24), f"Confidence: {row['confidence']:.3f}", fill="black")
    draw.text((5, CARD_H + 42), path.name[:48], fill="black")
    cards.append(card)

if cards:
    columns = 3
    rows = (len(cards) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * CARD_W, rows * (CARD_H + TEXT_H)), "#dddddd")
    for index, card in enumerate(cards):
        x = (index % columns) * CARD_W
        y = (index // columns) * (CARD_H + TEXT_H)
        sheet.paste(card, (x, y))
    display(sheet)

print("\nReview list saved to:", REVIEW_PATH)

Targeted validation errors: 17
      label predicted_label  confidence                                            image_path
Phytopthora           Fungi    0.999091 /content/potato_raw_fast/Phytopthora/potato_01028.jpg
Phytopthora           Fungi    0.999011 /content/potato_raw_fast/Phytopthora/potato_01070.jpg
      Virus         Healthy    0.997240       /content/potato_raw_fast/Virus/potato_02825.jpg
      Virus         Healthy    0.994912       /content/potato_raw_fast/Virus/potato_02824.jpg
    Healthy           Virus    0.994607     /content/potato_raw_fast/Healthy/potato_00774.jpg
      Virus         Healthy    0.993046       /content/potato_raw_fast/Virus/potato_03048.jpg
      Virus         Healthy    0.988557       /content/potato_raw_fast/Virus/potato_02612.jpg
Phytopthora           Fungi    0.987615 /content/potato_raw_fast/Phytopthora/potato_01025.jpg
Phytopthora           Fungi    0.981750 /content/potato_raw_fast/Phytopthora/potato_01087.jpg
Phytopthora           Fungi  

FileNotFoundError: [Errno 2] No such file or directory: '/content/potato_raw_fast/Phytopthora/potato_01028.jpg'

In [7]:
from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
PREDICTIONS_PATH = EXPERIMENT_DIR / "potato_validation_predictions_epoch14.csv"
CACHE_MANIFEST_PATH = EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"

predictions = pd.read_csv(PREDICTIONS_PATH)
cache_manifest = pd.read_csv(CACHE_MANIFEST_PATH)

# Match cached files by original filename; the cache folder is class-named.
cache_manifest["source_name"] = cache_manifest["image_path"].map(
    lambda path: Path(path).name
)
cache_lookup = dict(zip(cache_manifest["source_name"], cache_manifest["cache_path"]))

targeted = predictions[
    (~predictions["correct"])
    & (
        (predictions["label"] == "Healthy")
        | (
            (predictions["label"] == "Phytopthora")
            & (predictions["predicted_label"] == "Fungi")
        )
        | (
            (predictions["label"] == "Virus")
            & (predictions["predicted_label"].isin(["Healthy", "Fungi"]))
        )
    )
].copy()

targeted["cache_path"] = targeted["image_path"].map(
    lambda path: cache_lookup.get(Path(path).name)
)

missing = targeted[targeted["cache_path"].isna()]
if not missing.empty:
    print("Could not match these files to the cache:")
    print(missing["image_path"].to_string(index=False))

targeted = targeted[targeted["cache_path"].notna()].reset_index(drop=True)
print(f"Showing {len(targeted)} targeted validation errors using cached images.")

CARD_W, CARD_H, TEXT_H = 300, 300, 56
columns = 3
cards = []

for _, row in targeted.iterrows():
    cache_path = Path(row["cache_path"])
    if not cache_path.exists():
        print("Cached image missing:", cache_path)
        continue

    with Image.open(cache_path) as source:
        image = source.convert("RGB")
        image.thumbnail((CARD_W, CARD_H))

    card = Image.new("RGB", (CARD_W, CARD_H + TEXT_H), "white")
    card.paste(image, ((CARD_W - image.width) // 2, (CARD_H - image.height) // 2))
    draw = ImageDraw.Draw(card)
    draw.text((5, CARD_H + 4), f"True: {row['label']} | Pred: {row['predicted_label']}", fill="black")
    draw.text((5, CARD_H + 24), f"Confidence: {row['confidence']:.3f}", fill="black")
    draw.text((5, CARD_H + 41), Path(row["image_path"]).name, fill="black")
    cards.append(card)

if cards:
    rows = (len(cards) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * CARD_W, rows * (CARD_H + TEXT_H)), "#dddddd")
    for index, card in enumerate(cards):
        sheet.paste(card, ((index % columns) * CARD_W, (index // columns) * (CARD_H + TEXT_H)))
    display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [9]:
from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
BEST_CHECKPOINT_PATH = EXPERIMENT_DIR / "potato_best_model.pth"
PACKAGE_DIR = EXPERIMENT_DIR / "hf_package" / "potato" / "v1"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Best checkpoint missing: {BEST_CHECKPOINT_PATH}")

checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)
CLASS_NAMES = checkpoint["class_names"]

# Recreate the architecture and verify the checkpoint strictly.
class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet50(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.stem(images)
        x = self.layer1(x)

        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)

        output_size = fine.shape[-2:]
        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )
        deep = F.interpolate(
            self.project4(deep),
            size=output_size,
            mode="bilinear",
            align_corners=False,
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        attended = self.self_attention(fused)
        return self.pool(attended).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)

        batch_size, tile_count, channels, height, width = tile_images.shape
        flat_tiles = tile_images.reshape(
            batch_size * tile_count, channels, height, width
        )
        tile_features = self.encode_view(flat_tiles).reshape(
            batch_size, tile_count, -1
        )

        tile_weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1),
            dim=1,
        )
        weighted_tiles = (
            tile_features * tile_weights.unsqueeze(-1)
        ).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        combined = torch.cat(
            [full_features, weighted_tiles, strongest_tile],
            dim=1,
        )
        return self.classifier(combined)


model = PotatoResNet50Attention(num_classes=len(CLASS_NAMES))
load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()
print("Strict checkpoint load:", load_result)
print("Best checkpoint epoch:", checkpoint["epoch"])
print("Classes:", CLASS_NAMES)

# Save inference weights only, omitting optimizer and resume state.
weights_path = PACKAGE_DIR / "potato_model_v1_weights.pth"
torch.save(model.state_dict(), weights_path)

model_source = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet50(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)
        output_size = fine.shape[-2:]

        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=output_size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=output_size,
            mode="bilinear", align_corners=False
        )

        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)

        weights = torch.softmax(self.tile_score(tile_features).squeeze(-1), dim=1)
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        return self.classifier(
            torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)
        )
'''
(PACKAGE_DIR / "model.py").write_text(model_source.strip() + "\n")

inference_source = r'''
import json
from pathlib import Path

import torch
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import PotatoResNet50Attention


class PotatoDiseasePredictor:
    """Six-class Potato classifier. This package does not perform OOD rejection."""

    def __init__(self, model_dir=None, device=None):
        self.model_dir = Path(model_dir or Path(__file__).parent)
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        with open(self.model_dir / "config.json", "r", encoding="utf-8") as file:
            self.config = json.load(file)
        with open(self.model_dir / "classes.json", "r", encoding="utf-8") as file:
            self.classes = json.load(file)["classes"]

        self.image_size = int(self.config["image_size"])
        self.tile_size = int(self.config["tile_size"])
        self.mean = self.config["normalization"]["mean"]
        self.std = self.config["normalization"]["std"]

        self.model = PotatoResNet50Attention(len(self.classes)).to(self.device)
        state = torch.load(
            self.model_dir / self.config["weights_file"],
            map_location=self.device,
            weights_only=True,
        )
        self.model.load_state_dict(state, strict=True)
        self.model.eval()

        size, tile = self.image_size, self.tile_size
        self.tile_starts = [
            (0, 0),
            (size - tile, 0),
            (0, size - tile),
            (size - tile, size - tile),
        ]

    def _tensor(self, image):
        tensor = TF.to_tensor(image)
        return TF.normalize(tensor, self.mean, self.std)

    def _prepare(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = ImageOps.pad(
            image,
            (self.image_size, self.image_size),
            method=Image.Resampling.LANCZOS,
            color=(127, 127, 127),
        )
        full = self._tensor(image)
        tiles = torch.stack([
            self._tensor(image.crop(
                (left, top, left + self.tile_size, top + self.tile_size)
            ))
            for left, top in self.tile_starts
        ])
        return full.unsqueeze(0), tiles.unsqueeze(0)

    @torch.inference_mode()
    def predict(self, image, top_k=3):
        if isinstance(image, (str, Path)):
            with Image.open(image) as opened:
                image = opened.copy()

        full, tiles = self._prepare(image)
        full = full.to(self.device)
        tiles = tiles.to(self.device)
        logits = self.model(full, tiles)
        probabilities = torch.softmax(logits.float(), dim=1)[0]

        k = min(int(top_k), len(self.classes))
        values, indices = torch.topk(probabilities, k=k)
        top = [
            {"class": self.classes[int(index)], "confidence": float(value)}
            for value, index in zip(values.cpu(), indices.cpu())
        ]
        return {
            "crop": "potato",
            "decision": "PREDICTION",
            "predicted_class": top[0]["class"],
            "confidence": top[0]["confidence"],
            "top": top,
            "ood_rejection": False,
        }
'''
(PACKAGE_DIR / "inference.py").write_text(inference_source.strip() + "\n")

classes_config = {
    "crop": "potato",
    "classes": CLASS_NAMES,
}
(PACKAGE_DIR / "classes.json").write_text(json.dumps(classes_config, indent=2))

config = {
    "model_name": "PotatoResNet50Attention",
    "model_version": "1.0.0",
    "weights_file": "potato_model_v1_weights.pth",
    "image_size": 448,
    "tile_size": 288,
    "tile_count": 4,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
    "best_epoch": int(checkpoint["epoch"]),
    "validation_macro_f1": float(checkpoint["best_val_f1"]),
    "internal_test": {
        "images": 431,
        "accuracy": 0.8886,
        "balanced_accuracy": 0.8896,
        "macro_f1": 0.8807,
    },
    "ood_rejection_enabled": False,
}
(PACKAGE_DIR / "config.json").write_text(json.dumps(config, indent=2))

(PACKAGE_DIR / "requirements.txt").write_text(
    "torch>=2.1\n"
    "torchvision>=0.16\n"
    "Pillow>=9.0\n"
)

readme = f"""---
license: cc-by-4.0
---

# Potato Leaf Disease Classifier

Six-class image classifier for Potato leaf images. It uses a ResNet50 backbone,
multi-scale feature fusion, spatial self-attention, and four image tiles.

## Supported labels

{", ".join(CLASS_NAMES)}

The model expects one leaf image. Images are EXIF-corrected, padded to 448 × 448,
normalized with ImageNet mean/std, then evaluated as a full image plus four
288 × 288 corner tiles.

## Evaluation

- Best validation checkpoint: epoch {int(checkpoint["epoch"])}
- Validation macro-F1: {float(checkpoint["best_val_f1"]):.4f}
- Internal test set: 431 images
- Test accuracy: 0.8886
- Test balanced accuracy: 0.8896
- Test macro-F1: 0.8807

This package is a closed-set classifier and **does not include OOD rejection**.
It always returns one of the six supported labels. Do not interpret its output
as reliable for non-potato images or unsupported conditions.

## Usage

```python
from inference import PotatoDiseasePredictor

predictor = PotatoDiseasePredictor(model_dir=".")
result = predictor.predict("path/to/potato_leaf.jpg")
print(result)

_IncompleteInputError: incomplete input (3082528986.py, line 384)

In [10]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/potato_model_experiment"
)
PACKAGE_DIR = EXPERIMENT_DIR / "hf_package" / "potato" / "v1"
CHECKPOINT_PATH = EXPERIMENT_DIR / "potato_best_model.pth"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
CLASS_NAMES = checkpoint["class_names"]

class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(channels, num_heads, batch_first=True)
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(batch, channels, height, width)
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet50(weights=None)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1, self.layer2 = backbone.layer1, backbone.layer2
        self.layer3, self.layer4 = backbone.layer3, backbone.layer4
        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)
        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)
        output_size = fine.shape[-2:]

        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=output_size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=output_size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)
        weights = torch.softmax(self.tile_score(tile_features).squeeze(-1), dim=1)
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values
        return self.classifier(
            torch.cat([full_features, weighted_tiles, strongest_tile], dim=1)
        )


model = PotatoResNet50Attention(len(CLASS_NAMES))
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

torch.save(model.state_dict(), PACKAGE_DIR / "potato_model_v1_weights.pth")

model_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

''' + SpatialSelfAttention.__module__ if False else None

In [11]:
model_code = '''import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50


class SpatialSelfAttention(nn.Module):
    def __init__(self, channels=256, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        self.attention = nn.MultiheadAttention(
            channels, num_heads, batch_first=True
        )
        self.gate = nn.Conv2d(channels * 2, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, height, width = x.shape
        tokens = x.flatten(2).transpose(1, 2)
        normalized = self.norm(tokens)
        attended, _ = self.attention(
            normalized, normalized, normalized, need_weights=False
        )
        attended = attended.transpose(1, 2).reshape(
            batch, channels, height, width
        )
        gate = torch.sigmoid(self.gate(torch.cat([x, attended], dim=1)))
        return x + gate * attended


class PotatoResNet50Attention(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = resnet50(weights=None)

        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.project2 = nn.Conv2d(512, 256, kernel_size=1)
        self.project3 = nn.Conv2d(1024, 256, kernel_size=1)
        self.project4 = nn.Conv2d(2048, 256, kernel_size=1)

        self.fusion = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.self_attention = SpatialSelfAttention(256, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.tile_score = nn.Linear(256, 1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.35),
            nn.Linear(768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(512, num_classes),
        )

    def encode_view(self, images):
        x = self.layer1(self.stem(images))
        fine = self.layer2(x)
        middle = self.layer3(fine)
        deep = self.layer4(middle)
        output_size = fine.shape[-2:]

        fine = self.project2(fine)
        middle = F.interpolate(
            self.project3(middle), size=output_size,
            mode="bilinear", align_corners=False
        )
        deep = F.interpolate(
            self.project4(deep), size=output_size,
            mode="bilinear", align_corners=False
        )
        fused = self.fusion(torch.cat([fine, middle, deep], dim=1))
        return self.pool(self.self_attention(fused)).flatten(1)

    def forward(self, full_images, tile_images):
        full_features = self.encode_view(full_images)
        batch, count, channels, height, width = tile_images.shape
        tile_features = self.encode_view(
            tile_images.reshape(batch * count, channels, height, width)
        ).reshape(batch, count, -1)

        weights = torch.softmax(
            self.tile_score(tile_features).squeeze(-1), dim=1
        )
        weighted_tiles = (tile_features * weights.unsqueeze(-1)).sum(dim=1)
        strongest_tile = tile_features.max(dim=1).values

        return self.classifier(torch.cat(
            [full_features, weighted_tiles, strongest_tile], dim=1
        ))
'''

(PACKAGE_DIR / "model.py").write_text(model_code, encoding="utf-8")

print("Model definition saved:", PACKAGE_DIR / "model.py")
print("Inference weights saved:", (PACKAGE_DIR / "potato_model_v1_weights.pth").exists())

Model definition saved: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/hf_package/potato/v1/model.py
Inference weights saved: True


In [17]:
import json
from pathlib import Path

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/"
    "potato_model_experiment/hf_package/potato/v1"
)

classes = ["Bacteria", "Fungi", "Healthy", "Pest", "Phytopthora", "Virus"]

config = {
    "crop": "potato",
    "model_name": "PotatoResNet50Attention",
    "model_version": "1.0.0",
    "weights_file": "potato_model_v1_weights.pth",
    "image_size": 448,
    "tile_size": 288,
    "tile_count": 4,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
    "best_epoch": 14,
    "validation_macro_f1": 0.8791,
    "internal_test": {
        "images": 431,
        "accuracy": 0.8886,
        "balanced_accuracy": 0.8896,
        "macro_f1": 0.8807,
    },
    "ood_rejection_enabled": False,
}

(PACKAGE_DIR / "classes.json").write_text(
    json.dumps({"crop": "potato", "classes": classes}, indent=2),
    encoding="utf-8",
)
(PACKAGE_DIR / "config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)
(PACKAGE_DIR / "requirements.txt").write_text(
    "torch>=2.1\ntorchvision>=0.16\nPillow>=9.0\n",
    encoding="utf-8",
)

inference_code = '''import json
from pathlib import Path

import torch
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import PotatoResNet50Attention


class PotatoDiseasePredictor:
    """Six-class Potato classifier; this package does not reject OOD inputs."""

    def __init__(self, model_dir=None, device=None):
        self.model_dir = Path(model_dir or Path(__file__).parent)
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        with open(self.model_dir / "config.json", encoding="utf-8") as file:
            self.config = json.load(file)
        with open(self.model_dir / "classes.json", encoding="utf-8") as file:
            self.classes = json.load(file)["classes"]

        self.image_size = self.config["image_size"]
        self.tile_size = self.config["tile_size"]
        self.mean = self.config["normalization"]["mean"]
        self.std = self.config["normalization"]["std"]

        self.model = PotatoResNet50Attention(len(self.classes)).to(self.device)
        weights_path = self.model_dir / self.config["weights_file"]
        state = torch.load(
            weights_path, map_location=self.device, weights_only=True
        )
        self.model.load_state_dict(state, strict=True)
        self.model.eval()

        last = self.image_size - self.tile_size
        self.tile_starts = [(0, 0), (last, 0), (0, last), (last, last)]

    def _tensor(self, image):
        return TF.normalize(TF.to_tensor(image), self.mean, self.std)

    def _prepare(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = ImageOps.pad(
            image,
            (self.image_size, self.image_size),
            method=Image.Resampling.LANCZOS,
            color=(127, 127, 127),
        )
        full = self._tensor(image).unsqueeze(0)
        tiles = torch.stack([
            self._tensor(image.crop((
                left, top, left + self.tile_size, top + self.tile_size
            )))
            for left, top in self.tile_starts
        ]).unsqueeze(0)
        return full.to(self.device), tiles.to(self.device)

    @torch.inference_mode()
    def predict(self, image, top_k=3):
        if isinstance(image, (str, Path)):
            with Image.open(image) as opened:
                image = opened.copy()

        full, tiles = self._prepare(image)
        logits = self.model(full, tiles)
        probabilities = torch.softmax(logits.float(), dim=1)[0]
        count = min(int(top_k), len(self.classes))
        scores, indices = torch.topk(probabilities, count)

        top = [
            {"class": self.classes[int(index)], "confidence": float(score)}
            for score, index in zip(scores.cpu(), indices.cpu())
        ]
        return {
            "crop": "potato",
            "decision": "PREDICTION",
            "predicted_class": top[0]["class"],
            "confidence": top[0]["confidence"],
            "top": top,
            "ood_rejection": False,
        }
'''
(PACKAGE_DIR / "inference.py").write_text(inference_code, encoding="utf-8")

readme = """---
license: cc-by-4.0
---

# Potato Leaf Disease Classifier

Six-class Potato leaf image classifier using a ResNet50 backbone,
multi-scale feature fusion, spatial self-attention, and four image tiles.

## Supported classes

Bacteria, Fungi, Healthy, Pest, Phytopthora, Virus.

The inference wrapper applies EXIF orientation correction, pads the image to
448 x 448, and evaluates the full image plus four 288 x 288 corner tiles.

## Evaluation

- Best validation checkpoint: epoch 14
- Validation macro-F1: 0.8791
- Internal test set: 431 images
- Test accuracy: 0.8886
- Test balanced accuracy: 0.8896
- Test macro-F1: 0.8807

This is a closed-set classifier and does not include OOD rejection. It always
returns one of the six classes, so its predictions are not validated for
non-Potato images or unsupported Potato conditions.

## Usage

```python
from inference import PotatoDiseasePredictor

predictor = PotatoDiseasePredictor(model_dir=".")
result = predictor.predict("potato_leaf.jpg")
print(result)

_IncompleteInputError: incomplete input (2387860093.py, line 134)

In [18]:
import json
from pathlib import Path

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/"
    "potato_model_experiment/hf_package/potato/v1"
)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "crop": "potato",
    "model_name": "PotatoResNet50Attention",
    "model_version": "1.0.0",
    "weights_file": "potato_model_v1_weights.pth",
    "image_size": 448,
    "tile_size": 288,
    "tile_count": 4,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
    "classes": ["Bacteria", "Fungi", "Healthy", "Pest", "Phytopthora", "Virus"],
    "best_epoch": 14,
    "validation_macro_f1": 0.8791,
    "internal_test": {
        "images": 431,
        "accuracy": 0.8886,
        "balanced_accuracy": 0.8896,
        "macro_f1": 0.8807,
    },
    "ood_rejection_enabled": False,
}

(PACKAGE_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
(PACKAGE_DIR / "classes.json").write_text(
    json.dumps({"crop": "potato", "classes": config["classes"]}, indent=2),
    encoding="utf-8",
)
(PACKAGE_DIR / "requirements.txt").write_text(
    "torch>=2.1\ntorchvision>=0.16\nPillow>=9.0\n",
    encoding="utf-8",
)

readme_lines = [
    "---",
    "license: cc-by-4.0",
    "---",
    "",
    "# Potato Leaf Disease Classifier",
    "",
    "Six-class Potato leaf classifier using ResNet50, multi-scale feature fusion, spatial self-attention, and four image tiles.",
    "",
    "Classes: Bacteria, Fungi, Healthy, Pest, Phytopthora, Virus.",
    "",
    "## Evaluation",
    "- Best checkpoint: epoch 14",
    "- Validation macro-F1: 0.8791",
    "- Internal test set: 431 images",
    "- Test accuracy: 0.8886",
    "- Test balanced accuracy: 0.8896",
    "- Test macro-F1: 0.8807",
    "",
    "This is a closed-set classifier. OOD rejection is not included; it always predicts one of the six classes.",
    "",
    "## Dataset and citation",
    "Trained on a cleaned and split version of Project-AgML/potato_leaf_disease_classification (CC BY 4.0).",
    "Shabrina et al. (2024), A novel dataset of potato leaf disease in uncontrolled environment, Data in Brief, 52, 109955.",
    "https://doi.org/10.17632/ptz377bwb8.1",
    "",
    "The source dataset is not included in this package.",
]
(PACKAGE_DIR / "README.md").write_text("\n".join(readme_lines) + "\n", encoding="utf-8")

print("Saved metadata, requirements, and README.")
print("Package folder:", PACKAGE_DIR)
print("Files:", sorted(path.name for path in PACKAGE_DIR.iterdir()))

Saved metadata, requirements, and README.
Package folder: /content/drive/MyDrive/final_year_project/Potato/potato_model_experiment/hf_package/potato/v1
Files: ['README.md', 'classes.json', 'config.json', 'inference.py', 'model.py', 'potato_model_v1_weights.pth', 'requirements.txt']


In [19]:
import sys
from pathlib import Path
import pandas as pd

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/"
    "potato_model_experiment/hf_package/potato/v1"
)
EXPERIMENT_DIR = PACKAGE_DIR.parents[2]
CACHE_MANIFEST = EXPERIMENT_DIR / "potato_resized_cache_manifest.csv"

print("Inference file exists:", (PACKAGE_DIR / "inference.py").exists())
print("Model file exists:", (PACKAGE_DIR / "model.py").exists())
print("Weights exist:", (PACKAGE_DIR / "potato_model_v1_weights.pth").exists())

print("\nInference file beginning:")
print("\n".join((PACKAGE_DIR / "inference.py").read_text(encoding="utf-8").splitlines()[:12]))

sys.path.insert(0, str(PACKAGE_DIR))
from inference import PotatoDiseasePredictor

predictor = PotatoDiseasePredictor(model_dir=PACKAGE_DIR)

cache_df = pd.read_csv(CACHE_MANIFEST)
sample_path = cache_df.loc[cache_df["split"] == "internal_test", "cache_path"].iloc[0]
result = predictor.predict(sample_path)

print("\nSample used:", sample_path)
print("Prediction result:", result)

Inference file exists: True
Model file exists: True
Weights exist: True

Inference file beginning:
import json
from pathlib import Path

import torch
from PIL import Image, ImageOps
from torchvision.transforms import functional as TF

from model import PotatoResNet50Attention


class PotatoDiseasePredictor:
    """Six-class Potato classifier; this package does not reject OOD inputs."""

Sample used: /content/potato_cache_448/Fungi/potato_02237.jpg
Prediction result: {'crop': 'potato', 'decision': 'PREDICTION', 'predicted_class': 'Fungi', 'confidence': 0.999521017074585, 'top': [{'class': 'Fungi', 'confidence': 0.999521017074585}, {'class': 'Phytopthora', 'confidence': 0.00026234827237203717}, {'class': 'Pest', 'confidence': 0.00017443529213778675}], 'ood_rejection': False}


In [20]:
%pip -q install -U huggingface_hub

from huggingface_hub import notebook_login, HfApi
from pathlib import Path

PACKAGE_DIR = Path(
    "/content/drive/MyDrive/final_year_project/Potato/"
    "potato_model_experiment/hf_package/potato/v1"
)

required_files = [
    "README.md",
    "classes.json",
    "config.json",
    "inference.py",
    "model.py",
    "potato_model_v1_weights.pth",
    "requirements.txt",
]

missing = [name for name in required_files if not (PACKAGE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Package is missing files: {missing}")

print("Package ready. Total size (MB):",
      round(sum((PACKAGE_DIR / name).stat().st_size for name in required_files) / (1024**2), 2))

notebook_login()

api = HfApi()
upload_result = api.upload_folder(
    folder_path=str(PACKAGE_DIR),
    repo_id="virajwirle/plant-disease-models",
    repo_type="model",
    path_in_repo="potato/v1",
    allow_patterns=required_files,
    commit_message="Add Potato disease model v1",
)

print("Upload completed.")
print("Commit:", upload_result)
print("Model folder: https://huggingface.co/virajwirle/plant-disease-models/tree/main/potato/v1")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.
Package ready. Total size (MB): 103.28


Upload completed.
Commit: https://huggingface.co/virajwirle/plant-disease-models/commit/30e8c30c030c74713ae30d78ee85e2398713ff74
Model folder: https://huggingface.co/virajwirle/plant-disease-models/tree/main/potato/v1
